## attention is all you need

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from timm import create_model

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>attention is all you need.</p>

</div>



1. data preparation
===================

In [5]:
# define transformations for the dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resizing images to 224x224 - ViT expects larger images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

2. model
========
the function `timm.create_model` generates a vision transformer model (`vit_base_patch16_224`) using pre-trained weights from ImageNet. the value of `num_classes` set to 10 to align with the amount of classes in CIFAR-10.

In [6]:
# define Vision transformer Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = create_model('vit_base_patch16_224', pretrained=True, num_classes=10)  # a pretrained ViT model
model = model.to(device)

# loss Function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

3. trainig loop:
================
 handles input images in batches, executes a forward pass, computes the loss, and adjusts the model weights through backpropagation.

In [7]:
def train_model(model, train_loader, criterion, optimizer, device, epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct, total = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # zero the gradients
            optimizer.zero_grad()

            # forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(train_loader)}, Accuracy: {100 * correct / total:.2f}%')

4. assessment
==========================================
measure accuracy on classification

In [ ]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy on test data: {100 * correct / total:.2f}%')

# train and Evaluate the Model
train_model(model, train_loader, criterion, optimizer, device, epochs=1)
evaluate_model(model, test_loader, device)

## end